Basically the other notebook is a suboptimal to run on a computer. [MOST OF THE TRANSFORMER CODE FROM HERE](https://medium.com/data-science/build-your-own-transformer-from-scratch-using-pytorch-84c850470dcb)

In [31]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.tensorboard import SummaryWriter
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using: {device}")

data_path = "./Data_SecondaryStructure/data.csv"

seed = 42
np.random.seed(seed)
torch.manual_seed(seed)

Using: mps


In [32]:
df = pd.read_csv(data_path)
print(df.head())

  pdb_id chain_code  seq sst8 sst3  len  has_nonstd_aa
0   1A30          C  EDL  CBC  CEC    3          False
1   1B05          B  KCK  CBC  CEC    3          False
2   1B0H          B  KAK  CBC  CEC    3          False
3   1B1H          B  KFK  CBC  CEC    3          False
4   1B2H          B  KAK  CBC  CEC    3          False


In [33]:
df = df[df["len"] == 300]
print(f"Number of sequences with length == 300: {len(df)}")

Number of sequences with length == 300: 584


In [34]:
df = df[df["has_nonstd_aa"] == False]
df = df.drop(columns=["pdb_id", "chain_code", "sst3", "len", "has_nonstd_aa"])

print(df.head())
print(df.shape)


                                                      seq  \
268625  SVVEEHGQLSISNGELVNERGEQVQLKGMSSHGLQWYGQFVNYESM...   
268626  MDIFREIASSMKGENVFISPPSISSVLTILYYGANGSTAEQLSKYV...   
268627  TPPPTQWSYLCHPRVKEVQDEVDGYFLENWKFPSFKAVRTFLDAKF...   
268628  TPPPTQWSYLCHPRVKEVQDEVDGYFLENWKFPSFKAVRTFLDAKF...   
268629  TPPPTQWSYLCHPRVKEVQDEVDGYFLENWKFPSFKAVRTFLDAKF...   

                                                     sst8  
268625  CHHHHHCSCEEETTEEECTTSCBCCCEEEEESCHHHHGGGCSHHHH...  
268626  CHHHHHHHHHTTTSCEEECHHHHHHHHHHHHHHCCTHHHHHHHTTC...  
268627  CCCCCSCCCCCCTTHHHHHHHHHHHHHHHSCCSCHHHHHHHHHHCH...  
268628  CCCCCSCCCCCCTTHHHHHHHHHHHHHHHSCCSCHHHHHHHHHHCH...  
268629  CCCCCSCCCCCCTTHHHHHHHHHHHHHHHSCCSSHHHHHHHHHHCH...  
(574, 2)


In [35]:
AminoAcids = sorted(list(set("".join(df["seq"]))))
print(AminoAcids), len(AminoAcids)
SecondaryStructures = sorted(list(set("".join(df["sst8"]))))
print(SecondaryStructures), len(SecondaryStructures)

['A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y']
['B', 'C', 'E', 'G', 'H', 'I', 'S', 'T']


(None, 8)

In [36]:
aa_stoi = {s: i for i, s in enumerate(AminoAcids, start=1)}
aa_itos = {i: s for i, s in enumerate(AminoAcids, start=1)}
aa_encode = lambda s: [aa_stoi[c] for c in s]
aa_decode = lambda l: "".join([aa_itos[i] for i in l])

print(aa_encode("ACDEFGHIKLMNPQRSTVWY"))
print(aa_decode(aa_encode("ACDEFGHIKLMNPQRSTVWY")))

ss_stoi = {s: i for i, s in enumerate(SecondaryStructures, start=1)}
ss_itos = {i: s for i, s in enumerate(SecondaryStructures, start=1)}

PAD_LABEL = -100

ss_encode = lambda s: [ss_stoi[c] for c in s]
ss_decode = lambda l: "".join([ss_itos[i] for i in l if i != PAD_LABEL])

print(ss_encode("BCEGHIST"))
print(ss_decode(ss_encode("BCEGHIST")))

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
ACDEFGHIKLMNPQRSTVWY
[1, 2, 3, 4, 5, 6, 7, 8]
BCEGHIST


In [37]:
X = [torch.tensor(aa_encode(seq), dtype=torch.long) for seq in df["seq"]]
y = [torch.tensor(ss_encode(sst8), dtype=torch.long) for sst8 in df["sst8"]]

In [38]:
print(X[0]), print(y[0])
print(X[100]), print(y[100])

tensor([16, 18, 18,  4,  4,  7,  6, 14, 10, 16,  8, 16, 12,  6,  4, 10, 18, 12,
         4, 15,  6,  4, 14, 18, 14, 10,  9,  6, 11, 16, 16,  7,  6, 10, 14, 19,
        20,  6, 14,  5, 18, 12, 20,  4, 16, 11,  9, 19, 10, 15,  3,  3, 19,  6,
         8, 12, 18,  5, 15,  1,  1, 11, 20, 17, 16, 16,  6,  6, 20,  8,  3,  3,
        13, 16, 18,  9,  4,  9, 18,  9,  4,  1, 18,  4,  1,  1,  8,  3, 10,  3,
         8, 20, 18,  8,  8,  3, 19,  7,  8, 10, 16,  3, 12,  3, 13, 12,  8, 20,
         9,  4,  4,  1,  9,  3,  5,  5,  3,  4, 11, 16,  4, 10, 20,  6,  3, 20,
        13, 12, 18,  8, 20,  4,  8,  1, 12,  4, 13, 12,  6, 16,  3, 18, 17, 19,
         6, 12, 14,  8,  9, 13, 20,  1,  4,  4, 18,  8, 13,  8,  8, 15, 12, 12,
         3, 13, 12, 12,  8,  8,  8, 18,  6, 17,  6, 17, 19, 16, 14,  3, 18,  7,
         7,  1,  1,  3, 12, 14, 10,  1,  3, 13, 12, 18, 11, 20,  1,  5,  7,  5,
        20,  1,  6, 17,  7,  6, 14, 12, 10, 15,  3, 14, 18,  3, 20,  1, 10,  3,
        14,  6,  1,  1,  8,  5, 18, 16, 

(None, None)

In [39]:
from torch.utils.data import Dataset

class ProteinDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [40]:
dataset = ProteinDataset(X, y)
traindata, valdata, testdata = torch.utils.data.random_split(dataset, [0.80, 0.10, 0.10])

print(len(traindata), len(valdata), len(testdata))

460 57 57


In [41]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import math
import copy

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        
    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, -1e9)
        attn_probs = torch.softmax(attn_scores, dim=-1)
        output = torch.matmul(attn_probs, V)
        return output
        
    def split_heads(self, x):
        batch_size, seq_length, d_model = x.size()
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)
        
    def combine_heads(self, x):
        batch_size, _, seq_length, d_k = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_model)
        
    def forward(self, Q, K, V, mask=None):
        Q = self.split_heads(self.W_q(Q))
        K = self.split_heads(self.W_k(K))
        V = self.split_heads(self.W_v(V))
        
        attn_output = self.scaled_dot_product_attention(Q, K, V, mask)
        output = self.W_o(self.combine_heads(attn_output))
        return output

In [42]:
class PositionWiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super(PositionWiseFeedForward, self).__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

In [43]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_length):
        super(PositionalEncoding, self).__init__()
        
        pe = torch.zeros(max_seq_length, d_model)
        position = torch.arange(0, max_seq_length, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        self.register_buffer('pe', pe.unsqueeze(0))
        
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [44]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(EncoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, mask):
        attn_output = self.self_attn(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_output))
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))
        return x


In [45]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(DecoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, enc_output, src_mask, tgt_mask):
        attn_output = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout(attn_output))
        attn_output = self.cross_attn(x, enc_output, enc_output, src_mask)
        x = self.norm2(x + self.dropout(attn_output))
        ff_output = self.feed_forward(x)
        x = self.norm3(x + self.dropout(ff_output))
        return x

In [46]:
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model, num_heads, num_layers, d_ff, max_seq_length, dropout):
        super(Transformer, self).__init__()
        self.encoder_embedding = nn.Embedding(src_vocab_size, d_model)
        self.decoder_embedding = nn.Embedding(tgt_vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_length)

        self.encoder_layers = nn.ModuleList([EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.decoder_layers = nn.ModuleList([DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])

        self.fc = nn.Linear(d_model, tgt_vocab_size)
        self.dropout = nn.Dropout(dropout)

    def generate_mask(self, src, tgt):
        src_mask = (src != 0).unsqueeze(1).unsqueeze(2)
        tgt_mask = (tgt != 0).unsqueeze(1).unsqueeze(3)
        seq_length = tgt.size(1)
        nopeak_mask = torch.tril(torch.ones(1, seq_length, seq_length, device=tgt.device, dtype=torch.bool))
        tgt_mask = tgt_mask & nopeak_mask
        return src_mask, tgt_mask

    def forward(self, src, tgt):
        src_mask, tgt_mask = self.generate_mask(src, tgt)
        src_embedded = self.dropout(self.positional_encoding(self.encoder_embedding(src)))
        tgt_embedded = self.dropout(self.positional_encoding(self.decoder_embedding(tgt)))

        enc_output = src_embedded
        for enc_layer in self.encoder_layers:
            enc_output = enc_layer(enc_output, src_mask)

        dec_output = tgt_embedded
        for dec_layer in self.decoder_layers:
            dec_output = dec_layer(dec_output, enc_output, src_mask, tgt_mask)

        output = self.fc(dec_output)
        return output

In [47]:
src_vocab_size = len(AminoAcids) + 1  # +1 for padding
tgt_vocab_size = len(SecondaryStructures) + 1
d_model = 64
num_heads = 4
num_layers = 1
d_ff = 256
max_seq_length = 300
dropout = 0.2
batch_size = 16

transformer = Transformer(src_vocab_size, tgt_vocab_size, d_model, num_heads, num_layers, d_ff, max_seq_length, dropout)

src_data = torch.stack([
    torch.tensor(aa_encode(seq), dtype=torch.long)
    for seq in df["seq"]
])

tgt_data = torch.stack([
    torch.tensor(ss_encode(sst8), dtype=torch.long)
    for sst8 in df["sst8"]
])

In [48]:
criterion = nn.CrossEntropyLoss(ignore_index=-100)
optimizer = optim.Adam(transformer.parameters(), lr=0.0001, betas=(0.9, 0.98), eps=1e-9)

transformer = transformer.to(device)
src_data = src_data.to(device)
tgt_data = tgt_data.to(device)

transformer.train()

for epoch in range(100):
    optimizer.zero_grad()
    output = transformer(src_data, tgt_data[:, :-1])
    loss = criterion(output.contiguous().view(-1, tgt_vocab_size), tgt_data[:, 1:].contiguous().view(-1))
    loss.backward()
    optimizer.step()
    print(f"Epoch: {epoch+1}, Loss: {loss.item()}")

Epoch: 1, Loss: 2.6056196689605713
Epoch: 2, Loss: 2.5729331970214844
Epoch: 3, Loss: 2.537675380706787
Epoch: 4, Loss: 2.503833293914795
Epoch: 5, Loss: 2.4689180850982666
Epoch: 6, Loss: 2.4372904300689697
Epoch: 7, Loss: 2.4046976566314697
Epoch: 8, Loss: 2.371663808822632
Epoch: 9, Loss: 2.339503049850464
Epoch: 10, Loss: 2.306137800216675
Epoch: 11, Loss: 2.2766129970550537
Epoch: 12, Loss: 2.2447636127471924
Epoch: 13, Loss: 2.2151622772216797
Epoch: 14, Loss: 2.1837306022644043
Epoch: 15, Loss: 2.1545047760009766
Epoch: 16, Loss: 2.1263372898101807
Epoch: 17, Loss: 2.0970842838287354
Epoch: 18, Loss: 2.0707240104675293
Epoch: 19, Loss: 2.0423147678375244
Epoch: 20, Loss: 2.0158393383026123
Epoch: 21, Loss: 1.9921129941940308
Epoch: 22, Loss: 1.9674139022827148
Epoch: 23, Loss: 1.9427434206008911
Epoch: 24, Loss: 1.9172643423080444
Epoch: 25, Loss: 1.8951925039291382
Epoch: 26, Loss: 1.872955560684204
Epoch: 27, Loss: 1.8509080410003662
Epoch: 28, Loss: 1.8296047449111938
Epoch: 

In [49]:
torch.save(transformer.state_dict(), "./Data_SecondaryStructure/ss_transformer_trimmed.pth")

In [53]:
model = Transformer(src_vocab_size, tgt_vocab_size, d_model, num_heads, num_layers, d_ff, max_seq_length, dropout).to(device)
model.load_state_dict(torch.load("./Data_SecondaryStructure/ss_transformer_trimmed.pth", weights_only=True))

random_row = np.random.randint(0, len(testdata))
src_example = testdata[random_row][0].unsqueeze(0).to(device)
true_labels = testdata[random_row][1].unsqueeze(0).to(device)

# Teacher-forced decoder input: start token, then previous true labels.
start_token = torch.zeros((true_labels.size(0), 1), dtype=torch.long, device=device)
tgt_input = torch.cat([start_token, true_labels[:, :-1]], dim=1)

model.eval()
with torch.no_grad():
    prediction = model(src_example, tgt_input)
    predicted_labels = prediction.argmax(-1)
    accuracy = (predicted_labels == true_labels).float().mean().item()

predicted_sst8 = ss_decode(predicted_labels.squeeze(0).cpu().tolist())
true_sst8 = ss_decode(true_labels.squeeze(0).cpu().tolist())

print(f"Accuracy: {accuracy:.2%}")
print(f"Predicted: {predicted_sst8}")
print(f"True:      {true_sst8}")

Accuracy: 79.33%
Predicted: HCCCHHHHHHHHHHHHHHHTTCCHHHHHHHHHHHHHHHHHHHHHHHHHHHHHCCCHHCHHHHHHHHHHHHHHHHHHHHHHHCCHCHHHHHHHHHHHHHHCCCHHHHCHHHHHHHHHHHHHCHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHCCCHHHHHHHHHHHHCHHHHHHHHHHHHHCCHHHHTTCHHHHHHHHHHHHHHHHHHHHHHHHHHHHHTCCCCHHHHHHHHCTTHCHHHHHHHHHHHHHHHHHHHHHHHCCCTTHHHHHHHHHHCCHHHTTTTCCCCCCCCCCCCCCCC
True:      CCCHHHHHHHHHHHHTTSTTCCHHHHHHHHHHHHHHHHHHHHHHHHHHHHHCCCTTCHHHHHHHHHHHHHHHHHHHHHHHCCSCHHHHHHHHHHHHHTCCCSSTTCHHHHHHHHHHHHHCSSHHHHHHHHHHHHHHHHHHHHHHHHTTSSSCCCHHHHHHHHHHHTCHHHHHHHHHHHHTCCHHHHTTSHHHHHHHHHHHHHHHHHHHHHHHHHHHHHTCCCSHHHHHHHHCTTSCHHHHHHHHHHHHHHHHHHHHHHHCCCTTHHHHHHHHHHCCGGGTTTTCCCCCCCCCCCCCCCCC


In [ ]:
!TODO: Reorganize code into separate files/modules for data loading, model definition, training, and evaluation.

!TODO: Add more evaluation metrics (precision, recall, F1) and visualize predictions vs true labels.

!TODO: Implement proper training loop with batching, validation, and early stopping.

!TODO: Add learning rate scheduler and experiment with different hyperparameters (d_model, num_heads, num_layers, d_ff).

!TODO: Implement teacher forcing during training and evaluate on validation set after each epoch to monitor overfitting.

!TODO: Add positional encoding visualization to understand how the model learns sequence information.

zsh:1: command not found: TODO:
zsh:1: number expected
zsh:1: command not found: TODO:
zsh:1: no matches found: (d_model, num_heads, num_layers, d_ff).
zsh:1: command not found: TODO:
zsh:1: command not found: TODO:
